# MODUL PRAKTIKUM BIG DATA 2
## Pemrosesan Terstruktur Menggunakan PySpark DataFrame API & Spark SQL

Modul praktikum kedua ini berfokus pada pemrosesan data terstruktur skala besar menggunakan **Spark DataFrame API** dan **Spark SQL**. Praktikum ini menggunakan **NYC Yellow Taxi Trips Dataset** untuk melatih kemampuan data ingestion, transformasi waktu, agregasi, segmentasi kueri SQL, dan *window functions*.

---

### 📌 Dataset yang Digunakan
* **Nama Dataset:** NYC Yellow Taxi Trip Data
* **Deskripsi:** Dataset transaksi perjalanan taksi New York City yang memuat informasi lokasi/jam *pickup*, jarak tempuh, durasi, tarif, dan tips.

## 1. Unduh Dataset NYC Taxi ke DBFS / Colab

In [4]:
# 1. Unduh file dataset alternatif yang aktif (format terkompresi .csv.gz)
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_2021-01.csv.gz -O /tmp/nyc_taxi.csv.gz

# 2. Ekstrak file gz tersebut menjadi file csv murni
!gunzip -f /tmp/nyc_taxi.csv.gz

# 3. Pindahkan file hasil ekstrak ke direktori tiruan DBFS di Colab
!cp /tmp/nyc_taxi.csv /dbfs/FileStore/tables/nyc_taxi.csv

# 4. Verifikasi keberadaan dan ukuran file
!ls -lh /dbfs/FileStore/tables/nyc_taxi.csv

--2026-11-08 05:29:15--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/f6895842-79e6-4a43-9458-e5b0b454a340?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-11-08T06%3A20%3A03Z&rscd=attachment%3B+filename%3Dyellow_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-11-08T05%3A19%3A59Z&ske=2026-11-08T06%3A20%3A03Z&sks=b&skv=2018-11-09&sig=hJDk9R%2Bw5l%2FzkiZQyIcu8NqVapO3BRAscX1XSOd8kcs%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc5NDExNzU1NSwibmJmIjoxNzk0MTE1NzU1LCJwY

In [5]:
from pyspark.sql import SparkSession

# 1. Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Praktikum_NYCTaxi_Luthfi") \
    .master("local[*]") \
    .getOrCreate()

# 2. Path file dataset yang sudah kita siapkan
file_path = "/dbfs/FileStore/tables/nyc_taxi.csv"

# 3. Membaca dataset CSV ke dalam DataFrame
df_taxi = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

# 4. Verifikasi: Tampilkan skema tipe data dan 5 baris pertama
print("--- SKEMA DATAFRAME ---")
df_taxi.printSchema()

print("\n--- 5 BARIS PERTAMA ---")
df_taxi.show(5, truncate=False)

--- SKEMA DATAFRAME ---
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)


--- 5 BARIS PERTAMA ---
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+-----

## 2. Ingestion Data dengan Explicit Schema

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Skema eksplisit disesuaikan letaknya agar urutan kolom CSV tidak bergeser,
# namun tetap mempertahankan nama kolom (seperti vendor_id, pickup_datetime) sesuai modul Anda.
schema = StructType([
    StructField("vendor_id", StringType(), True),
    StructField("pickup_datetime", StringType(), True),
    StructField("dropoff_datetime", StringType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("rate_code_id", StringType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", IntegerType(), True),      # <-- Sisipan agar tidak geser
    StructField("DOLocationID", IntegerType(), True),      # <-- Sisipan agar tidak geser
    StructField("payment_type", StringType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True), # <-- Sisipan agar tidak geser
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True)   # <-- Sisipan agar tidak geser
])

# Gunakan absolute path lokal di Colab (tanpa awalan dbfs:/)
file_path = "/dbfs/FileStore/tables/nyc_taxi.csv"

# Membaca dataset dengan skema eksplisit
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load(file_path)

print("--- SKEMA SPARK DATAFRAME ---")
df_raw.printSchema()
print(f"Total Record Data: {df_raw.count():,}")

--- SKEMA SPARK DATAFRAME ---
root
 |-- vendor_id: string (nullable = true)
 |-- pickup_datetime: string (nullable = true)
 |-- dropoff_datetime: string (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- rate_code_id: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)

Total Record Data: 1,369,765


## 3. Pembersihan Data & Feature Engineering

In [7]:
# Cleaning & Datetime Extraction
df_clean = df_raw.dropna(subset=["pickup_datetime", "total_amount"]) \
    .filter((F.col("trip_distance") > 0) & (F.col("total_amount") > 0))

# Perbaikan: Menghapus strip di awal baris dan menambahkan operator (-) di perhitungan durasi
df_transformed = df_clean \
    .withColumn("pickup_ts", F.to_timestamp("pickup_datetime", "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("dropoff_ts", F.to_timestamp("dropoff_datetime", "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("pickup_hour", F.hour("pickup_ts")) \
    .withColumn("trip_duration_min", F.round((F.unix_timestamp("dropoff_ts") - F.unix_timestamp("pickup_ts")) / 60, 2))

# Mendaftarkan Temporary View untuk Kueri Spark SQL
df_transformed.createOrReplaceTempView("v_nyc_taxi")

print("--- CONTOH DATA TERTATA (5 BARIS) ---")
df_transformed.select("vendor_id", "pickup_hour", "trip_distance", "trip_duration_min", "total_amount").show(5)

--- CONTOH DATA TERTATA (5 BARIS) ---
+---------+-----------+-------------+-----------------+------------+
|vendor_id|pickup_hour|trip_distance|trip_duration_min|total_amount|
+---------+-----------+-------------+-----------------+------------+
|        1|          0|          2.1|             6.03|        11.8|
|        1|          0|          0.2|             0.98|         4.3|
|        1|          0|         14.7|             27.6|       51.95|
|        1|          0|         10.6|            15.22|       36.35|
|        2|          0|         4.94|            16.53|       24.36|
+---------+-----------+-------------+-----------------+------------+
only showing top 5 rows


## 4. Agregasi DataFrame API (Contoh Demonstrasi)

In [8]:
# 4. Agregasi DataFrame API (Contoh Demonstrasi)
df_hourly_summary = df_transformed.groupBy("pickup_hour") \
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue")
    ) \
    .sort(F.desc("total_trips"))

print("--- RINGKASAN PERJALANAN PER JAM ---")
df_hourly_summary.show(5)

--- RINGKASAN PERJALANAN PER JAM ---
+-----------+-----------+------------+-------------+
|pickup_hour|total_trips|avg_distance|total_revenue|
+-----------+-----------+------------+-------------+
|         15|     111152|        3.23|    1900495.6|
|         14|     110446|        4.75|   1873726.94|
|         16|     104812|        2.83|   1888702.54|
|         17|     102850|        2.93|   1867686.27|
|         13|     102377|        5.68|   1685057.87|
+-----------+-----------+------------+-------------+
only showing top 5 rows


---

## 5. TUGAS MANDIRI MAHASISWA (SPARK DATAFRAME & SPARK SQL)

Kerjakan tugas-tugas di bawah ini menggunakan **Spark DataFrame API** atau **Spark SQL** pada sel kode di bawah!

---


### 5. TUGAS MANDIRI MAHASISWA: SPARK DATAFRAME & SPARK SQL

#### Dataset: NYC Taxi Trips Dataset ('v_nyc_taxi' / 'df_transformed')
#### Petunjuk: Tuliskan jawaban Anda pada sel yang telah disediakan!
---

# ------------------------------------------------------------------------------
### KATEGORI A: ANALISIS TREN WAKTU & METRIK PERJALANAN (DATAFRAME API)
# ------------------------------------------------------------------------------


In [10]:
# TUGAS A1: Analisis Jam Sibuk Penumpang (Peak Pick-up Hours)
# Instruksi:
# 1. Kelompokkan data berdasarkan `pickup_hour` (Jam 00 s.d. 23).
# 2. Hitung:
#    - Total frekuensi perjalanan (Count)
#    - Rata-rata jarak tempuh (`trip_distance`)
#    - Total pendapatan (`total_amount`)
# 3. Urutkan berdasarkan total frekuensi perjalanan terbanyak dan tampilkan Top 5 Jam Sibuk!
#
# Tulis kode Anda di bawah ini:
# ================= TUGAS A1 =================

# 1. Kelompokkan berdasarkan jam (pickup_hour)
# 2. Hitung Total Frekuensi, Rata-rata Jarak, dan Total Pendapatan
# 3. Urutkan berdasarkan frekuensi terbanyak (descending)
df_tugas_a1 = df_transformed.groupBy("pickup_hour") \
    .agg(
        F.count("*").alias("total_frekuensi"),
        F.round(F.avg("trip_distance"), 2).alias("rata_rata_jarak"),
        F.round(F.sum("total_amount"), 2).alias("total_pendapatan")
    ) \
    .orderBy(F.desc("total_frekuensi"))

print("--- TUGAS A1: TOP 5 JAM SIBUK PENUMPANG ---")
df_tugas_a1.show(5)


--- TUGAS A1: TOP 5 JAM SIBUK PENUMPANG ---
+-----------+---------------+---------------+----------------+
|pickup_hour|total_frekuensi|rata_rata_jarak|total_pendapatan|
+-----------+---------------+---------------+----------------+
|         15|         111152|           3.23|       1900495.6|
|         14|         110446|           4.75|      1873726.94|
|         16|         104812|           2.83|      1888702.54|
|         17|         102850|           2.93|      1867686.27|
|         13|         102377|           5.68|      1685057.87|
+-----------+---------------+---------------+----------------+
only showing top 5 rows


In [12]:
# TUGAS A2: Analisis Durasi Perjalanan Berdasarkan Jumlah Penumpang
# Instruksi:
# 1. Kelompokkan data berdasarkan `passenger_count`.
# 2. Hitung rata-rata durasi perjalanan (`trip_duration_min`) dan rata-rata total tarif (`total_amount`).
# 3. Filter kelompok penumpang yang lebih dari 0 dan urutkan berdasarkan jumlah penumpang.
#
# Tulis kode Anda di bawah ini:
# ================= TUGAS A2 =================

# 1. Filter penumpang > 0, lalu kelompokkan berdasarkan passenger_count
# 2. Hitung rata-rata durasi dan tarif
# 3. Urutkan berdasarkan jumlah penumpang
df_tugas_a2 = df_transformed.filter(F.col("passenger_count") > 0) \
    .groupBy("passenger_count") \
    .agg(
        F.round(F.avg("trip_duration_min"), 2).alias("rata_rata_durasi_menit"),
        F.round(F.avg("total_amount"), 2).alias("rata_rata_total_tarif")
    ) \
    .orderBy("passenger_count")

print("--- TUGAS A2: ANALISIS DURASI & TARIF BERDASARKAN PENUMPANG ---")
df_tugas_a2.show()

--- TUGAS A2: ANALISIS DURASI & TARIF BERDASARKAN PENUMPANG ---
+---------------+----------------------+---------------------+
|passenger_count|rata_rata_durasi_menit|rata_rata_total_tarif|
+---------------+----------------------+---------------------+
|              1|                 12.78|                16.36|
|              2|                 14.19|                17.05|
|              3|                 15.39|                16.81|
|              4|                  15.3|                17.22|
|              5|                 21.31|                16.55|
|              6|                  15.5|                16.22|
|              7|                  25.0|               101.71|
|              8|                 19.85|                 16.3|
+---------------+----------------------+---------------------+



# ------------------------------------------------------------------------------
### KATEGORI B: ANALISIS SQL & SEGMENTASI TARIF (SPARK SQL)
# ------------------------------------------------------------------------------

In [13]:
# TUGAS B1: Segmentasi Jarak Tempuh Menggunakan CASE WHEN di Spark SQL
# Instruksi:
# Tuliskan kueri SQL dengan `spark.sql()` untuk:
# 1. Mengategorikan `trip_distance` ke dalam 3 kategori:
#    - 'Short Trip' : trip_distance <= 2.0
#    - 'Medium Trip': trip_distance BETWEEN 2.1 AND 10.0
#    - 'Long Trip'  : trip_distance > 10.0
# 2. Hitung jumlah total perjalanan, rata-rata ongkos (`fare_amount`), dan rata-rata tips (`tip_amount`) per segmen!
#
# Tulis kode Anda di bawah ini:
# ================= TUGAS B1 =================

# Mengeksekusi kueri menggunakan Spark SQL
df_tugas_b1 = spark.sql("""
    SELECT
        CASE
            WHEN trip_distance <= 2.0 THEN 'Short Trip'
            WHEN trip_distance > 2.0 AND trip_distance <= 10.0 THEN 'Medium Trip'
            ELSE 'Long Trip'
        END AS trip_segment,
        COUNT(*) AS total_trips,
        ROUND(AVG(fare_amount), 2) AS avg_fare,
        ROUND(AVG(tip_amount), 2) AS avg_tip
    FROM v_nyc_taxi
    GROUP BY
        CASE
            WHEN trip_distance <= 2.0 THEN 'Short Trip'
            WHEN trip_distance > 2.0 AND trip_distance <= 10.0 THEN 'Medium Trip'
            ELSE 'Long Trip'
        END
    ORDER BY total_trips DESC
""")

print("--- TUGAS B1: SEGMENTASI JARAK TEMPUH ---")
df_tugas_b1.show()

--- TUGAS B1: SEGMENTASI JARAK TEMPUH ---
+------------+-----------+--------+-------+
|trip_segment|total_trips|avg_fare|avg_tip|
+------------+-----------+--------+-------+
|  Short Trip|     779134|    6.77|   1.44|
| Medium Trip|     492818|   15.52|   2.38|
|   Long Trip|      71692|   46.22|    4.3|
+------------+-----------+--------+-------+



In [14]:
# TUGAS B2: Analisis Metode Pembayaran & Tip Rate
# Instruksi:
# Tuliskan kueri SQL untuk:
# 1. Kelompokkan data berdasarkan `payment_type`.
# 2. Hitung rata-rata persentase tip (`(tip_amount / total_amount) * 100`).
# 3. Tampilkan tipe pembayaran yang memberikan rata-rata tip tertinggi!
#
# Tulis kode Anda di bawah ini:
# ================= TUGAS B2 =================

# Mengeksekusi kueri menggunakan Spark SQL
df_tugas_b2 = spark.sql("""
    SELECT
        payment_type,
        ROUND(AVG((tip_amount / total_amount) * 100), 2) AS avg_tip_percentage
    FROM v_nyc_taxi
    GROUP BY payment_type
    ORDER BY avg_tip_percentage DESC
""")

print("--- TUGAS B2: ANALISIS METODE PEMBAYARAN & TIP RATE ---")
df_tugas_b2.show()

--- TUGAS B2: ANALISIS METODE PEMBAYARAN & TIP RATE ---
+------------+------------------+
|payment_type|avg_tip_percentage|
+------------+------------------+
|           1|             15.47|
|        NULL|              6.52|
|           3|              0.01|
|           4|               0.0|
|           2|               0.0|
+------------+------------------+



# ------------------------------------------------------------------------------
### KATEGORI C: ADVANCED WINDOW FUNCTIONS & EXPORT
# ------------------------------------------------------------------------------

In [15]:
# TUGAS C1: Ranking Perjalanan Terjauh Per Jam Menggunakan Window Function
# Instruksi:
# 1. Gunakan Window Function (`Window.partitionBy("pickup_hour").orderBy(F.desc("trip_distance"))`).
# 2. Berikan ranking (`dense_rank()`) pada setiap perjalanan berdasarkan jarak tempuh di setiap jamnya.
# 3. Tampilkan perjalanan urutan ke-1 (terjauh) untuk setiap jam pada jam 08:00, 12:00, dan 18:00!
#
# Tulis kode Anda di bawah ini:
# ================= TUGAS C1 =================
from pyspark.sql.window import Window

# 1. Definisikan partisi Window Function
window_spec = Window.partitionBy("pickup_hour").orderBy(F.desc("trip_distance"))

# 2. Terapkan dense_rank() dan hitung rankingnya
# 3. Filter hanya ranking 1 (terjauh) pada jam 8, 12, dan 18
df_tugas_c1 = df_transformed \
    .withColumn("jarak_rank", F.dense_rank().over(window_spec)) \
    .filter(F.col("jarak_rank") == 1) \
    .filter(F.col("pickup_hour").isin(8, 12, 18)) \
    .select("pickup_hour", "trip_distance", "total_amount", "jarak_rank") \
    .orderBy("pickup_hour")

print("--- TUGAS C1: PERJALANAN TERJAUH DI JAM 08:00, 12:00, DAN 18:00 ---")
df_tugas_c1.show()

--- TUGAS C1: PERJALANAN TERJAUH DI JAM 08:00, 12:00, DAN 18:00 ---
+-----------+-------------+------------+----------+
|pickup_hour|trip_distance|total_amount|jarak_rank|
+-----------+-------------+------------+----------+
|          8|     112257.1|       17.66|         1|
|         12|     51676.03|       38.84|         1|
|         18|         59.9|      310.56|         1|
+-----------+-------------+------------+----------+



In [16]:
# TUGAS C2: Export Hasil Agregasi ke Parquet Format
# Instruksi:
# 1. Simpan DataFrame hasil agregasi jam sibuk (Tugas A1) ke direktori DBFS dalam format terkompresi **Parquet**.
# 2. Lokasi target: `dbfs:/FileStore/tables/output/nyc_hourly_summary.parquet`
# 3. Mode penyimpanan: 'overwrite'.
#
# Tulis kode Anda di bawah ini:
# ================= TUGAS C2 =================

# 2. Lokasi target disesuaikan untuk sistem lokal Colab
output_path = "/dbfs/FileStore/tables/output/nyc_hourly_summary.parquet"

# 1 & 3. Menyimpan df_tugas_a1 ke format Parquet dengan mode overwrite
df_tugas_a1.write \
    .mode("overwrite") \
    .parquet(output_path)

print("--- TUGAS C2: EXPORT PARQUET BERHASIL ---")
print(f"Data agregasi jam sibuk telah berhasil disimpan di: {output_path}")

# (Opsional) Verifikasi file yang telah dibuat
import os
if os.path.exists(output_path):
    print("Verifikasi: File/Folder Parquet ditemukan!")

--- TUGAS C2: EXPORT PARQUET BERHASIL ---
Data agregasi jam sibuk telah berhasil disimpan di: /dbfs/FileStore/tables/output/nyc_hourly_summary.parquet
Verifikasi: File/Folder Parquet ditemukan!
